# V119 - Exact chat-twin transfer

Pre-registered plan: 1182 chat texts are exact duplicates across train and test, so some test users speak identically to train users. Transfer the mean labels of exact-chat twins from training folds only, fallback to word-only Ridge without twins. Same grouped folds under a fresh seed, same locks. If identical words meant identical needs, this wins big.

In [1]:
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 4848
N_SPLITS = 5
GATE_FLOOR = 0.002
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v119")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Chat-twin means inside training folds, then gate

Twins share the full concatenated chat document exactly. Matches use fold-train users only.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in list(train["user_id"])]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()

from sklearn.model_selection import StratifiedKFold as SKF
strata = Y.argmax(1)
fold_groups = np.zeros(n_train, int)
for fi, (tr, va) in enumerate(SKF(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(docs, strata)):
    fold_groups[va] = fi
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

oof_w, oof_t, fids = np.zeros_like(Y), np.zeros_like(Y), np.zeros(n_train, int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(W, strata, fold_groups)):
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(W[tr], Y[tr, j])
        oof_w[va, j] = m.predict(W[va])
    pool = defaultdict(list)
    for u in tr:
        pool[docs[u]].append(u)
    for u in va:
        mate = [v for v in pool[docs[u]] if v != u]
        oof_t[u] = Y[mate].mean(0) if mate else oof_w[u]
    fids[va] = fi + 1
base = ndcg5(Y, oof_w)
d = ndcg5(Y, oof_t) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fids == k], oof_t[fids == k]).mean() - base[fids == k].mean() for k in range(1, N_SPLITS + 1))
print(f"word={base.mean():.5f} chat-twin={ndcg5(Y, oof_t).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT chat-twin transfer" if (lo > GATE_FLOOR and wf >= 0) else "KEEP baseline; identical words do not imply identical needs")

word=0.64301 chat-twin=0.62764 delta=-0.01537 CI95=[-0.01799,-0.01268] worst-fold=-0.01988
KEEP baseline; identical words do not imply identical needs
